# Семинар 6. За счёт чего изменилась выручка

Легенда:
Выручка ПРАЙМ от подписок осенью 2025 года: сентябрь → октябрь **+7,1 %**,
октябрь → ноябрь **−0,3 %**, ноябрь → декабрь **+6,7 %**.

Финансовый директор: «Росли на семь процентов и встали. Что сломалось в ноябре?»

У нас одна пара и один слайд для него.

Метки те же, что на третьем семинаре:

| Метка | Что значит |
|---|---|
| **понимать** | спросят на защите и на проверочной |
| **уметь писать** | понадобится в проекте |
| **знать, что существует** | достаточно помнить, что такое бывает, и загуглить |


---

## 1. Данные и вопрос

Одна готовая выгрузка по всей базе: месяц × тариф × канал привлечения. В строке —
активные подписки, число успешных платежей и выручка. Строк 56, числа у всех
одинаковые. Запрос идёт 5–10 секунд.

Подписка **активна в месяце**, если началась до его конца и не закончилась до его
начала, — как на Л5. Канал берём у владельца подписки. Сутки и месяцы считаем
по UTC: это делает `set time zone 'UTC'` внутри `q()`.

Если сервер не отвечает — третья ячейка кода ниже хранит запасную таблицу
с теми же строками.


In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC — так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])


In [ ]:
def load_monthly(client_filter: str = "true", first: str = "2025-09-01", last: str = "2025-12-01") -> pd.DataFrame:
    """Месяц × тариф × канал: активные подписки, число платежей, выручка."""
    return q(f"""
        with seg as (
            select client_id, acquisition_channel as channel
            from prime.clients
            where {client_filter}
        ),
        act as (
            select to_char(m.month_start, 'YYYY-MM') as month, s.tariff_id, c.channel, count(*) as active
            from generate_series(cast(:first as timestamptz), cast(:last as timestamptz), interval '1 month') as m(month_start)
            join prime.subscriptions s
              on s.started_at < m.month_start + interval '1 month'
             and (s.ended_at is null or s.ended_at >= m.month_start)
            join seg c on c.client_id = s.owner_client_id
            group by 1, 2, 3
        ),
        rev as (
            select to_char(p.paid_at, 'YYYY-MM') as month, s.tariff_id, c.channel,
                   count(*) as n_payments, sum(p.amount) as revenue
            from prime.payments p
            join prime.subscriptions s on s.subscription_id = p.subscription_id
            join seg c on c.client_id = s.owner_client_id
            where p.status = 'success'
              and p.paid_at >= cast(:first as timestamptz)
              and p.paid_at < cast(:last as timestamptz) + interval '1 month'
            group by 1, 2, 3
        )
        select a.month, a.tariff_id, a.channel, a.active,
               coalesce(r.n_payments, 0) as n_payments, coalesce(r.revenue, 0) as revenue
        from act a
        left join rev r
          on r.month = a.month and r.tariff_id = a.tariff_id and r.channel is not distinct from a.channel
        order by 1, 2, 3
    """, first=first, last=last)


monthly = load_monthly()
monthly_raw = monthly.copy()      # нетронутая копия: пригодится в задаче ★4

print(len(monthly), "строк")


In [ ]:
# Запасной выход: здесь те же 56 строк, что возвращает запрос выше.
# Если сервер не отвечает, поменяйте False на True и запустите эту ячейку.
# Пока стоит False, ячейка ничего не меняет.
SERVER_IS_DOWN = False

backup = pd.DataFrame([
    ("2025-09", 1, "crm", 11667, 10983, 2185617.0),
    ("2025-09", 1, "organic", 26158, 24741, 4923459.0),
    ("2025-09", 1, "partner_telecom", 16997, 16035, 3190965.0),
    ("2025-09", 1, "performance", 18878, 17796, 3541404.0),
    ("2025-09", 1, "referral", 8109, 7673, 1526927.0),
    ("2025-09", 1, "social", 12196, 11522, 2292878.0),
    ("2025-09", 1, None, 1940, 1827, 363573.0),
    ("2025-09", 2, "crm", 7385, 6951, 2773449.0),
    ("2025-09", 2, "organic", 15927, 15088, 6020112.0),
    ("2025-09", 2, "partner_telecom", 10182, 9604, 3831996.0),
    ("2025-09", 2, "performance", 11248, 10585, 4223415.0),
    ("2025-09", 2, "referral", 5135, 4868, 1942332.0),
    ("2025-09", 2, "social", 7415, 6952, 2773848.0),
    ("2025-09", 2, None, 1198, 1125, 448875.0),
    ("2025-10", 1, "crm", 12135, 11789, 2346011.0),
    ("2025-10", 1, "organic", 27281, 26609, 5295191.0),
    ("2025-10", 1, "partner_telecom", 17607, 17190, 3420810.0),
    ("2025-10", 1, "performance", 19454, 18931, 3767269.0),
    ("2025-10", 1, "referral", 8454, 8281, 1647919.0),
    ("2025-10", 1, "social", 12547, 12192, 2426208.0),
    ("2025-10", 1, None, 1989, 1951, 388249.0),
    ("2025-10", 2, "crm", 7665, 7434, 2966166.0),
    ("2025-10", 2, "organic", 16600, 16263, 6488937.0),
    ("2025-10", 2, "partner_telecom", 10559, 10270, 4097730.0),
    ("2025-10", 2, "performance", 11605, 11299, 4508301.0),
    ("2025-10", 2, "referral", 5331, 5217, 2081583.0),
    ("2025-10", 2, "social", 7630, 7386, 2947014.0),
    ("2025-10", 2, None, 1242, 1204, 480396.0),
    ("2025-11", 1, "crm", 12482, 11791, 2346409.0),
    ("2025-11", 1, "organic", 28326, 26728, 5318872.0),
    ("2025-11", 1, "partner_telecom", 18129, 17195, 3421805.0),
    ("2025-11", 1, "performance", 19864, 18715, 3724285.0),
    ("2025-11", 1, "referral", 8746, 8263, 1644337.0),
    ("2025-11", 1, "social", 12757, 12029, 2393771.0),
    ("2025-11", 1, None, 2036, 1922, 382478.0),
    ("2025-11", 2, "crm", 7919, 7514, 2998086.0),
    ("2025-11", 2, "organic", 17209, 16228, 6474972.0),
    ("2025-11", 2, "partner_telecom", 10882, 10241, 4086159.0),
    ("2025-11", 2, "performance", 11865, 11176, 4459224.0),
    ("2025-11", 2, "referral", 5525, 5240, 2090760.0),
    ("2025-11", 2, "social", 7791, 7283, 2905917.0),
    ("2025-11", 2, None, 1275, 1190, 474810.0),
    ("2025-12", 1, "crm", 12922, 12602, 2507798.0),
    ("2025-12", 1, "organic", 29404, 28775, 5726225.0),
    ("2025-12", 1, "partner_telecom", 18704, 18245, 3630755.0),
    ("2025-12", 1, "performance", 20391, 19821, 3944379.0),
    ("2025-12", 1, "referral", 9059, 8826, 1756374.0),
    ("2025-12", 1, "social", 13068, 12709, 2529091.0),
    ("2025-12", 1, None, 2098, 2062, 410338.0),
    ("2025-12", 2, "crm", 8177, 7973, 3181227.0),
    ("2025-12", 2, "organic", 17843, 17419, 6950181.0),
    ("2025-12", 2, "partner_telecom", 11304, 10976, 4379424.0),
    ("2025-12", 2, "performance", 12226, 11899, 4747701.0),
    ("2025-12", 2, "referral", 5720, 5592, 2231208.0),
    ("2025-12", 2, "social", 7930, 7690, 3068310.0),
    ("2025-12", 2, None, 1317, 1292, 515508.0),
], columns=["month", "tariff_id", "channel", "active", "n_payments", "revenue"])

if SERVER_IS_DOWN:
    monthly, monthly_raw = backup.copy(), backup.copy()
    print("взяли запасную таблицу:", len(monthly), "строк")


In [ ]:
monthly["tariff"] = monthly["tariff_id"].map({1: "Базовый", 2: "Расширенный"})

monthly.head(8)


In [ ]:
# Выручка по месяцам и её прирост к предыдущему месяцу, в процентах.
# На Л5 прирост считали как x / x.shift(1) - 1. Какой метод делает это сам?
revenue = monthly.groupby("month")["revenue"].sum()
growth = revenue.ЗАПОЛНИТЕ() * 100

print(pd.DataFrame({"revenue": revenue, "growth_pct": growth.round(1)}))


**+7,1 %, −0,3 %, +6,7 %.** Директор прав: в ноябре рост пропал.

Прежде чем считать — гипотезы. Вырос отток? Упали продажи? Отвалился канал?
Запишите свою: в конце пары посмотрим, какая выжила.


---

## 2. Где: аддитивная модель по каналам

Выручка — сумма выручек каналов. Значит, её изменение — сумма изменений каналов.
Это **аддитивная модель**, слайды 5–6: она отвечает на вопрос «где».


In [ ]:
# Каналы — в строки, месяцы — в колонки. В каждую клетку попадают две строки
# исходной таблицы, по одной на тариф. Что с ними сделать?
by_channel = monthly.pivot_table(index="channel", columns="month",
                                 values="revenue", aggfunc=ЗАПОЛНИТЕ)
by_channel["delta"] = by_channel["2025-11"] - by_channel["2025-10"]
total_delta = revenue["2025-11"] - revenue["2025-10"]

print(by_channel[["2025-10", "2025-11", "delta"]].round().astype(int))
print()
print(f"сумма по каналам: {by_channel['delta'].sum():>10,.0f} ₽".replace(",", " "))
print(f"изменение итога:  {total_delta:>10,.0f} ₽".replace(",", " "))


**Не сходится на 11 357 ₽.** Каналы в сумме дают −128 542 ₽, а выручка изменилась
на −139 899 ₽. В аддитивной модели сумма частей равна целому до копейки. Значит,
часть потерялась.

Куда делись одиннадцать тысяч?


In [ ]:
# Все ли строки чьи-то? Считаем каналы вместе с пропусками — как на третьем семинаре.
print(monthly["channel"].value_counts(dropna=False))

no_channel = monthly[monthly["channel"].isna()].groupby("month")["revenue"].sum()
print()
print("выручка клиентов без канала:")
print(no_channel[["2025-10", "2025-11"]].astype(int).to_string())


**Восемь строк без канала.** `pivot_table` молча выбросила строки с пропуском
в ключе — так же, как `groupby` и `value_counts` на третьем семинаре. Клиенты без
канала принесли 868 645 ₽ в октябре и 857 288 ₽ в ноябре: вот они, 11 357 ₽.

Это та самая ошибка с графика демо на лекции — «каналы не сходятся с итогом»,
слайды 3–4. Теперь вы сделали её своими руками.


In [ ]:
# Что делать с клиентами без канала: выбросить, отнести в «прочее»
# или показать отдельной строкой? Чем заполняем пропуск?
monthly["channel"] = monthly["channel"].fillna(ЗАПОЛНИТЕ)

print("пропусков в канале:", monthly["channel"].isna().sum())


In [ ]:
by_channel = monthly.pivot_table(index="channel", columns="month",
                                 values="revenue", aggfunc="sum")
by_channel["delta"] = by_channel["2025-11"] - by_channel["2025-10"]
by_channel = by_channel.sort_values("delta")

# Та же сверка, но теперь она обязана сойтись: сумма частей равна целому.
assert abs(by_channel["delta"].sum() - total_delta) < 0.01, "каналы не сходятся с итогом"

summary = by_channel[["2025-10", "2025-11", "delta"]].copy()
summary.loc["итого"] = summary.sum()
summary.round().astype(int)


Договорённость курса: клиентов без канала не выбрасываем и не прячем в «прочее»,
а показываем отдельной строкой — «без канала».

Теперь сходится. И одного виноватого нет: четыре строки в минусе, три в плюсе.

**performance и social вместе дают −166 тыс. ₽ при итоге −140 тыс.** Это «118 %
падения» — живой пример слайда 16: когда части тянут в разные стороны, доли
выходят за сто процентов. Поэтому показывают рубли.

**Уметь писать.** `pivot_table`, колонка `delta` и `assert` на сумму — с этих трёх
строк начинается любое разложение. Без `aggfunc` таблица тоже построится, но
посчитает среднее по двум тарифам, а не сумму, — и не скажет об этом.


In [ ]:
# Активная база по каналам: на сколько процентов она выросла за тот же месяц.
active = monthly.pivot_table(index="channel", columns="month", values="active", aggfunc="sum")
base_growth = (active["2025-11"] / active["2025-10"] - 1) * 100

base_growth.sort_values().round(1)


Медленнее всех растут базы social и performance: +1,8 % и +2,2 %. У четырёх других
каналов — от +3,0 до +3,8 %. Это те самые два канала, у которых на семинаре 5 был
самый высокий отток.


In [ ]:
# То же по тарифам.
by_tariff = monthly.pivot_table(index="tariff", columns="month", values="revenue", aggfunc="sum")
by_tariff["delta"] = by_tariff["2025-11"] - by_tariff["2025-10"]

by_tariff[["2025-10", "2025-11", "delta"]].round().astype(int)


Оба тарифа в минусе: −60 тыс. и −80 тыс. ₽. Значит, дело не в тарифе.

Аддитивная модель ответила «где»: почти везде и понемногу. Чтобы понять,
**за счёт чего**, выручку надо не складывать, а раскладывать на множители.

> **Зачем это в работе.** «Продажи упали — в каком регионе, магазине, продукте?»
> Любой разбор начинают со сложения по частям: если виновата одна, дальше можно
> не считать.


---

## 3. За счёт чего: активные × выручка на подписку

**Мультипликативная модель**, слайды 7–8: показатель — произведение факторов.

> выручка = активные подписки × выручка на одну активную подписку

Второй множитель — это ARPU с Л5, у которого в знаменателе активные подписки.


In [ ]:
m = monthly.groupby("month")[["revenue", "n_payments", "active"]].sum()
m["revenue_per_sub"] = m["revenue"] / m["active"]

m.round(2)


In [ ]:
def chain(n0, p0, n1, p1):
    """Цепные подстановки для R = N × P: сначала количество, потом цена."""
    by_count = (n1 - n0) * p0          # сначала количество — при старой цене
    # Теперь цена. Количество уже новое — на что его умножаем?
    by_price = n1 * (ЗАПОЛНИТЕ)
    assert abs(by_count + by_price - (n1 * p1 - n0 * p0)) < 0.01, "сумма влияний не равна изменению"
    return round(by_count), round(by_price)


# Пекарня с лекции: было 100 булок по 50 ₽, стало 90 булок по 60 ₽.
# Какие два числа должны получиться? Назовите их ДО запуска.
print(chain(100, 50, 90, 60))


In [ ]:
for a, b in (("2025-09", "2025-10"), ("2025-10", "2025-11")):
    by_active, by_revenue_per_sub = chain(m.loc[a, "active"], m.loc[a, "revenue_per_sub"],
                                          m.loc[b, "active"], m.loc[b, "revenue_per_sub"])
    print(f"{a} → {b}:   активные подписки {by_active:>+10,} ₽"
          f"   выручка на подписку {by_revenue_per_sub:>+10,} ₽".replace(",", " "))


База растёт оба раза: **+1,47 млн** и **+1,26 млн ₽**. А выручка на подписку
в октябре дала **+1,35 млн**, в ноябре — **−1,40 млн**: весь рост базы она съела.

Цены в эти месяцы не менялись. **Что тогда упало?**

**Уметь писать.** `chain` с `assert` внутри: ошибка упадёт сразу, а не в отчёте.
Порядок — сначала количество, потом цена — пишут в подписи к разложению.

> **Зачем это в работе.** Выручка магазина — число покупок × сумма покупки, доход
> по кредитам — портфель × ставка. Где показатель — произведение, вопрос «за счёт
> чего» решается так же.


---

## 4. Три фактора

«Выручка на подписку» — два числа в одном: сколько раз подписка заплатила за месяц
и сколько стоил один платёж.

> выручка = активные подписки × платежей на подписку × цена платежа

Два фактора — формула. Три — уже цикл, слайд 13.


In [ ]:
m["per_sub"] = m["n_payments"] / m["active"]      # платежей на одну активную подписку
m["price"] = m["revenue"] / m["n_payments"]       # цена одного платежа

LABELS = {"active": "активные подписки", "per_sub": "платежей на подписку", "price": "цена платежа"}

m[["active", "per_sub", "price"]].round(4)


In [ ]:
def contrib(x0: dict, x1: dict, order: list[str]) -> dict:
    """Влияние каждого фактора: меняем их по одному в порядке order."""
    cur, out = dict(x0), {}
    for f in order:
        before = math.prod(cur.values())
        # Один фактор переходит на новое значение, остальные стоят. На какое?
        cur[f] = ЗАПОЛНИТЕ
        out[f] = math.prod(cur.values()) - before
    total = math.prod(x1.values()) - math.prod(x0.values())
    assert abs(sum(out.values()) - total) < 0.01, "сумма влияний не равна изменению"
    return out


# Игрушка с лекции: комиссия = операции × чек операции × ставка.
# Ждём 3000, −3300 и 1980.
toy = contrib({"ops": 1000, "chk": 2000, "rate": 0.015},
              {"ops": 1100, "chk": 1800, "rate": 0.016},
              ["ops", "chk", "rate"])
print({f: round(v) for f, v in toy.items()})


In [ ]:
x0 = m.loc["2025-10", ["active", "per_sub", "price"]].to_dict()
x1 = m.loc["2025-11", ["active", "per_sub", "price"]].to_dict()

effects = pd.Series(contrib(x0, x1, ["active", "per_sub", "price"]))
assert abs(effects.sum() - total_delta) < 0.01, "сумма влияний не равна изменению выручки"

print(effects.rename(LABELS).round().astype(int).to_string())
print()
print(f"сумма влияний {effects.sum():,.0f} ₽ — изменение выручки {total_delta:,.0f} ₽".replace(",", " "))


**Цена ни при чём: −2 тыс. ₽.** Тарифы не дешевели — Базовый стоит 199 ₽,
Расширенный 399 ₽ в оба месяца. Чуть сдвинулась доля тарифов в платежах: это
эффект структуры, слайд 15 и задача ★1.

Упало число платежей: **0,9745 → 0,9436** на одну активную подписку.

Подписка стала платить реже? Нет. Списание идёт **раз в 30 дней** — а в октябре
31 день, в ноябре 30. Кто заплатил первого октября, заплатил ещё раз тридцать
первого. Отношение 0,9745 / 0,9436 = 1,033 — почти те же «1,032 платежа
на подписку», что на первой паре: там был июль, в нём тоже 31 день.


In [ ]:
# Сколько дней в каждом месяце? У периода есть готовое свойство — слайд 18 Л5.
m["days"] = pd.PeriodIndex(m.index, freq="M").ЗАПОЛНИТЕ
m["revenue_per_day"] = m["revenue"] / m["days"]
m["growth_per_day"] = m["revenue_per_day"].pct_change() * 100

m[["revenue", "days", "revenue_per_day", "growth_per_day"]].round(1)


**Рост не останавливался.** Выручка в день росла ровно: **+3,6 %, +3,0 %, +3,2 %**.

Ноябрь короче октября на день — вот и вся «остановка». Отсюда и «+7,1 %»,
и «−0,3 %»: длинный месяц после короткого, потом короткий после длинного.
Ни одна гипотеза из начала пары не понадобилась: ни отток, ни продажи, ни канал.

**Понимать.** Разложение ответило «за счёт чего»: число платежей на подписку.
«Почему» оно не знает. Про 30 дней между списаниями и про длину месяца знаете
вы, а не формула: арифметика показывает, где искать, причину называет знание
о бизнесе.

> **Зачем это в работе.** Февраль против января, квартал с праздниками против
> квартала без них: периоды разной длины сравнивают в пересчёте на день, иначе
> каждый короткий выглядит как авария.


---

## 5. Каскад для директора

Форму графика выбирают под вывод, слайд 18. Наш вывод — «из чего сложилось
изменение», значит, каскадная диаграмма.

На лекции каскад шёл от уровня к уровню: выручка была → изменения → выручка стала.
Здесь так не выйдет: шаг в 1,26 млн на фоне 42,9 млн при оси от нуля — несколько
пикселей, а обрезать ось у столбиков нельзя, слайд 22.

Поэтому рисуем каскад **изменений**, в тысячах рублей. Столбики факторов «плывут»:
каждый начинается там, где закончился предыдущий. Последний столбик — «итого»,
он стоит на нуле. Ось включает ноль.


In [ ]:
def waterfall(deltas: pd.Series, title: str) -> plt.Axes:
    """Каскад изменений: по столбику на фактор и «итого» в конце."""
    before = deltas.cumsum() - deltas      # уровень, на котором закончился предыдущий столбик
    colors = ["#C1272D" if v < 0 else "#2A9D8F" for v in deltas]

    fig, ax = plt.subplots(figsize=(9, 4.6), layout="constrained")
    # С какого уровня стартует столбик фактора: с нуля — или с другого места?
    bars = ax.bar(deltas.index, deltas, bottom=ЗАПОЛНИТЕ, color=colors, zorder=2)
    total = ax.bar("итого", deltas.sum(), color="#6B7280", zorder=2)
    for group in (bars, total):
        ax.bar_label(group, fmt=lambda v: f"{v:+,.0f}".replace(",", " ").replace("-", "−"), padding=3)
    ax.axhline(0, color="#111827", linewidth=0.8)
    ax.set_title(title, loc="left")
    ax.set_ylabel("тыс. ₽")
    ax.use_sticky_edges = False            # иначе подписи упрутся в край графика
    ax.margins(y=0.15)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#E5E7EB")
    return ax


steps = effects.rename(LABELS) / 1000
# Заголовок графика — это вывод, слайд 18. Что вы скажете директору одной строкой?
ax = waterfall(steps, title=ЗАПОЛНИТЕ)
ax.set_xlabel("Октябрь → ноябрь 2025. Порядок подстановок — слева направо.")
plt.show()


Проверьте свой график по лекции: заголовок говорит вывод, ось включает ноль,
единицы подписаны, период и порядок подстановок названы в подписи.

Столбик «цена платежа» почти не виден — и это правильно: его высота честно
показывает, что цена ни при чём.

**Уметь писать.** `ax.bar(..., bottom=...)` — вся каскадная диаграмма в одном
аргументе. Остальное — оформление.

> **Зачем это в работе.** Каскад — обычная форма план-факт анализа: «план → цена
> → объём → курс → факт». Его читают без пояснений.


---

## 6. Вывод за 30 секунд

Директор спросит не «как вы считали», а «что случилось и что делать». Схема
с лекции, слайд 28, — четыре абзаца.

Напишите их здесь своими словами и с числами из ноутбука: дважды щёлкните
по ячейке и замените многоточия.

**Что случилось** — число, период и с чем сравнили:

…

**За счёт чего** — разложение:

…

**Чего мы не знаем:**

…

**Что делать** — действие, кто, когда:

…


---

## 7. Если успеем, иначе дома

Два продолжения. Первое проверяет разгадку на всех переходах сразу, второе —
обещанная на лекции комиссия.

### Все переходы сразу

Если дело в длине месяца, влияние выручки на подписку должно менять знак
от перехода к переходу, а влияние базы — нет.


In [ ]:
# Чтобы разложить все переходы разом, рядом с каждым месяцем нужен предыдущий.
# На сколько строк сдвигаем таблицу?
prev = m.shift(ЗАПОЛНИТЕ)

all_steps = pd.DataFrame({
    "by_active": (m["active"] - prev["active"]) * prev["revenue_per_sub"],
    "by_revenue_per_sub": m["active"] * (m["revenue_per_sub"] - prev["revenue_per_sub"]),
}).dropna()

gap_to_total = all_steps.sum(axis=1) - m["revenue"].diff().dropna()
assert gap_to_total.abs().max() < 0.01, "сумма влияний не равна изменению выручки"

(all_steps / 1000).round()      # тыс. ₽; строка — месяц, в который пришли


In [ ]:
chart = (all_steps / 1000).rename(columns={"by_active": "активные подписки",
                                           "by_revenue_per_sub": "выручка на подписку"})
chart.index = [f"{month}, {days} дн." for month, days in m.loc[chart.index, "days"].items()]

fig, ax = plt.subplots(figsize=(9, 4.2), layout="constrained")
chart.plot.bar(ax=ax, stacked=True, color=["#9CA3AF", "#C1272D"], rot=0)
ax.axhline(0, color="#111827", linewidth=0.8)
ax.set_title("Выручка на подписку меняет знак вместе с длиной месяца", loc="left")
ax.set_ylabel("тыс. ₽")
ax.set_xlabel("Месяц, в который пришли. Порядок подстановок: сначала активные подписки.")
ax.spines[["top", "right"]].set_visible(False)
plt.show()


Влияние базы — три плюса подряд. Влияние выручки на подписку — плюс, минус, плюс:
месяц в 31 день после месяца в 30 — плюс, наоборот — минус. Серый цвет — фон,
красный — то, о чём вывод: слайд 23. На всём 2025 годе это проверяет задача ★7.

### Комиссия = число операций × чек операции × ставка

Второй доход ПРАЙМ — комиссия с покупок в партнёрских сервисах. Число операций
и оборот в базе есть. **Ставки в базе нет:** она лежит в договоре с партнёрами,
а не в таблице.

Тогда множитель задают **допущением** и пишут об этом в подписи. Наше допущение:
1,5 % в октябре и 1,4 % в ноябре.


In [ ]:
fees = q("""
    select to_char(occurred_at, 'YYYY-MM') as month, count(*) as ops, sum(amount) as turnover
    from prime.transactions
    where status = 'success' and service_id is not null and amount > 0
      and occurred_at >= '2025-10-01' and occurred_at < '2025-12-01'
    group by 1 order by 1
""").set_index("month")

RATE = {"2025-10": 0.015, "2025-11": 0.014}       # допущение: ставки в базе нет

fees["chk"] = fees["turnover"] / fees["ops"]      # чек операции
fees["rate"] = pd.Series(RATE)
fees["fee"] = fees["ops"] * fees["chk"] * fees["rate"]

fees.round({"turnover": 0, "chk": 2, "fee": 0})


In [ ]:
f0 = fees.loc["2025-10", ["ops", "chk", "rate"]].to_dict()
f1 = fees.loc["2025-11", ["ops", "chk", "rate"]].to_dict()

# В каком порядке меняем факторы? Договорённость с лекции: сначала «сколько», потом «почём».
fee_effects = contrib(f0, f1, order=ЗАПОЛНИТЕ)

print({f: round(v) for f, v in fee_effects.items()})
print("изменение комиссии:", round(fees.loc["2025-11", "fee"] - fees.loc["2025-10", "fee"]), "₽")


In [ ]:
import itertools

# Все шесть порядков: строка — порядок, колонка — фактор.
orders = pd.DataFrame({" → ".join(order): contrib(f0, f1, list(order))
                       for order in itertools.permutations(["ops", "chk", "rate"])}).T
orders.loc["среднее по порядкам"] = orders.mean()

orders.round()


**Ставка дала −146 тыс. ₽ при общем изменении −124 тыс.** Главный вывод разложения
целиком стоит на числе, которого в данных нет. Другое допущение — другой вывод,
и читатель должен это видеть.

Шесть порядков — шесть ответов, сумма одна. Различаются они на совместные члены:
здесь это не больше трёх тысяч рублей на фактор.

**Знать, что существует.** Среднее по всем порядкам от порядка не зависит. Его
берут, когда о порядке не договориться; в учебниках оно называется вектором Шепли.

> **Зачем это в работе.** Ставка, курс, себестоимость: в выгрузке почти всегда
> не хватает одного множителя. Его задают допущением и называют в подписи.


---

## 8. Компонент: `src/prime/factors.py`

Три функции с этой пары переезжают в ваш репозиторий — как проверки на третьем
семинаре.

Создайте файл `src/prime/factors.py` и перенесите туда `chain`, `contrib`
и `waterfall`. Сверху — импорты, которые им нужны:

```python
import math

import matplotlib.pyplot as plt
import pandas as pd
```

После этого в любом ноутбуке внутри `prime-monitor` работает:

```python
from prime.factors import chain, contrib, waterfall
```

Коммит — в `main`, одной строкой:

```
git add src/prime/factors.py && git commit -m "факторное разложение с семинара 6"
```

Файл пригодится в Проекте 1.


---

## Что унести с сегодняшнего дня

1. **Сумма влияний равна изменению — и это проверяет `assert`.** Не сошлось
   на 11 357 ₽ — значит, что-то потеряли: `pivot_table` молча выбросила клиентов
   без канала.
2. **Сначала «где», потом «за счёт чего».** По каналам и тарифам — сложением,
   внутри — умножением.
3. **Цепные подстановки меняют факторы по одному.** Порядок — часть ответа,
   его пишут в подписи.
4. **Месяц — не единица измерения.** В пересчёте на день выручка росла ровно:
   +3,6 %, +3,0 %, +3,2 %.
5. **Разложение отвечает «за счёт чего», а не «почему».** Число платежей
   на подписку назвала арифметика, длину месяца — вы.
6. **Заголовок графика — это вывод.** У столбиков ось включает ноль, влияния
   показывают в рублях, а не в долях.
7. **Вывод — четыре абзаца:** что случилось, за счёт чего, чего не знаем,
   что делать.

### Где вы запустили то, что было на лекции

| Слайд Л6 | Приём | Где |
|---|---|---|
| 2 | сумма влияний равна изменению | `assert` в разделах 2, 3, 4 и 7 |
| 3–4 | три ошибки графика демо | раздел 2 — каналы не сходятся с итогом; раздел 4 — периоды разной длины; раздел 5 — ось |
| 5–6 | аддитивная модель | раздел 2 |
| 7–8 | мультипликативная модель | раздел 3 |
| 9, 12 | цепные подстановки, функция `chain` | раздел 3 |
| 10–11 | порядок подстановок, совместный член | раздел 7 — шесть порядков; ★2 и ★3 |
| 13 | три фактора циклом | раздел 4, `contrib`; раздел 7 — комиссия |
| 14 | сочетание моделей | задача 3 хвоста |
| 15 | эффект структуры | раздел 4 — «цена платежа»; ★1 |
| 16 | доли больше 100 % | раздел 2 — «118 % падения»; ★5 |
| 18 | сначала вывод, потом форма графика | раздел 5 — заголовок пишете вы |
| 19–21 | каскадная диаграмма | раздел 5, `waterfall` |
| 22 | у столбиков ось с нуля | раздел 5 — каскад изменений вместо каскада уровней |
| 23 | один цвет для вывода | раздел 7 — график всех переходов |
| 24 | неполный период | раздел 4 — выручка в день; задача 3 хвоста — равные окна; ★7 |
| 25 | типовые искажения | раздел 5 — ось; остальные четыре — лекция, слайд 25 |
| 26 | деловой график | раздел 5 — оформление `waterfall` |
| 28 | вывод за 30 секунд | раздел 6 |
| 29 | чего разложение не доказывает | раздел 4, «понимать»; раздел 6, «чего мы не знаем» |
| 30 | Проект 1 | ниже |

### Проект 1

Четыре опоры — те же, что на слайде 30:

- сумма влияний сходится с изменением, и это проверено кодом;
- порядок подстановок и окна сравнения названы в подписи;
- каждый из 3–5 графиков что-то доказывает, и заголовок говорит вывод;
- записка идёт по схеме из четырёх шагов: что случилось → за счёт чего → чего
  не знаем → что делать.

Гипотеза, которая не подтвердилась, — тоже результат, и в записке ей место:
сегодня не пригодилась ни одна из трёх.

**Сдача — 25.10 в 23:59, защиты — 26.10–06.11.**

### Ваш вопрос с первого семинара

На первом семинаре каждый записал бизнес-вопрос к данным ПРАЙМ. Достаньте его
и ответьте на три вопроса:

1. Какой показатель в нём меняется?
2. На какие части он **складывается**? Это ответ «где».
3. На какие множители он **раскладывается**? Это ответ «за счёт чего».

Есть ответы на второй и третий — есть план разложения.


---

## Хвост: ваш сегмент

Дальше — работа с открытым концом: сколько успеете на паре, остальное дома.
Ничего из этого не сдаётся.

Сегмент — тот же, что в ДЗ-2: условие лежит в вашей строке `prime.assignments`,
в колонке `client_filter`. Числа у каждого свои, поэтому проверки по хешу здесь нет.


In [ ]:
my_filter = q("select client_filter from prime.assignments where login = current_user").iloc[0, 0]
mine = load_monthly(my_filter)

print(len(mine), "строк")
mine.head(3)


In [ ]:
# Задача 1. Разложите октябрь → ноябрь своего сегмента на активные подписки
# и выручку на подписку: таблица mine устроена так же, как monthly.
# Совпал ли знак с общим? На малом сегменте шум сравним с самим эффектом.
my_effects = None   # ← ваш расчёт

print(my_effects)


In [ ]:
# Задача 2. Каскад изменений по каналам своего сегмента, октябрь → ноябрь.
# Функция waterfall уже есть; заголовок — ваш вывод. Не забудьте клиентов без канала.
# Если подписи каналов наезжают друг на друга: ax.tick_params(axis="x", labelrotation=20)
my_deltas = None   # ← ваш расчёт и ваш график

print(my_deltas)


### Задача 3. Дно

Пример с лекции — на вашем сегменте. Последняя неделя августа 2026 года против
предыдущей, в равных окнах с понедельника по субботу: 17–22.08 и 24–29.08.

Запрос готов и отдаёт то же, что таблица `win` на слайде 6: канал, окно `w`,
число платежей `n` и выручку `r`. Разложение — за вами: изменение каждого канала,
а внутри канала — влияние числа платежей и цены платежа. Это слайды 6, 8 и 14:
сложение по каналам, умножение внутри.

Задача намеренно трудная. В ней спрятан вопрос: на лекции почти всё падение дал
один канал — а в вашем сегменте он вообще есть?


In [ ]:
# Задача 3. Окна w0 (17–22.08) и w1 (24–29.08) готовым запросом — дальше сами.
win = q(f"""
    with seg as (
        select client_id, acquisition_channel as channel
        from prime.clients
        where {my_filter}
    )
    select case when p.paid_at < '2026-08-23' then 'w0' else 'w1' end as w,
           c.channel, count(*) as n, sum(p.amount) as r
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    join seg c on c.client_id = s.owner_client_id
    where p.status = 'success'
      and ((p.paid_at >= '2026-08-17' and p.paid_at < '2026-08-23')
        or (p.paid_at >= '2026-08-24' and p.paid_at < '2026-08-30'))
    group by 1, 2
    order by 1, 2
""")

my_august = None   # ← ваш расчёт

print(my_august)


---

## Задачи со звёздочкой

Семь задач на то, что на лекции было на экране, а руками вы этого не делали.
**Не сдаются и не оцениваются.** Зачем тогда: в Проекте 1 расчёт метрик
и факторное разложение — 35 % оценки, и ошибаться дешевле здесь.

Считаются на общей таблице `monthly` — она у всех одинаковая — и опираются
на `m`, `chain` и `contrib` из разделов 1–4. Если перезапускали ядро, сначала
прогоните эти разделы.

Проверка знает ответ, но не показывает его: замените `None` своим расчётом
и запустите ячейку. Формат ответа задан в условии — без него хеш не сойдётся.


In [ ]:
# Это просто запустите
ANSWERS = {
    "★1": "18acb87648",
    "★2": "0c181642eb",
    "★3": "978fb791c8",
    "★4": "719ba01dac",
    "★5": "106b449a5a",
    "★6": "66bb771226",
    "★7": "89d8dad468"
}


In [ ]:
# Это тоже
import hashlib

import numpy as np
import pandas as pd


def _norm(x):
    if isinstance(x, bool):
        return repr(x)
    if isinstance(x, float):
        return f"{x:.2f}"
    if isinstance(x, int):
        return str(x)
    if isinstance(x, dict):
        return "{" + ",".join(f"{_norm(k)}:{_norm(v)}" for k, v in sorted(x.items(), key=lambda kv: _norm(kv[0]))) + "}"
    if isinstance(x, (set, frozenset)):
        return "{" + ",".join(sorted(_norm(v) for v in x)) + "}"
    if isinstance(x, (list, tuple)):
        return "[" + ",".join(_norm(v) for v in x) + "]"
    if isinstance(x, np.integer):
        return str(int(x))
    if isinstance(x, np.bool_):
        return repr(bool(x))
    if isinstance(x, pd.Timestamp):
        return x.isoformat()
    if isinstance(x, pd.Timedelta):
        return str(int(x.total_seconds()))
    if isinstance(x, pd.Index):
        return _norm(list(x))
    if isinstance(x, pd.Series):
        return "{" + ",".join(f"{_norm(k)}:{_norm(v)}" for k, v in sorted(x.items(), key=lambda kv: _norm(kv[0]))) + "}"
    if isinstance(x, pd.DataFrame):
        d = x if isinstance(x.index, pd.RangeIndex) else x.reset_index()
        cols = sorted(map(str, d.columns))
        rows = sorted(tuple(_norm(r[c]) for c in cols) for _, r in d.iterrows())
        return "[" + ",".join("[" + ",".join(r) + "]" for r in rows) + "]"
    return repr(x)


def _show(x) -> str:
    """Короткий показ ответа: кадр на сто тысяч строк в вывод не печатаем."""
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return f"{type(x).__name__} {x.shape}\n{x.head().to_string()}"
    return repr(x)


def check(task: str, got) -> None:
    """Сверить ответ с эталоном. Эталоны лежат хешами — подсмотреть нельзя."""
    if got is None:
        print(f"{task}: не решено")
        return
    digest = hashlib.sha256(_norm(got).encode()).hexdigest()[:10]
    if not ANSWERS:
        print(f"HASH:{task}={digest}")
    elif digest == ANSWERS.get(task):
        print(f"{task}: верно")
    else:
        print(f"{task}: не сходится, получилось {_show(got)}")


print("проверка готова")


### ★ 1. Эффект структуры: цена растёт без изменения цен

Слайд 15. Два тарифа по 199 и 399 ₽, цены не меняются. Меняется только доля
тарифов в платежах — она задана в ячейке.

На сколько рублей изменилась средняя цена платежа?

Так в разделе 4 и получились −2 тыс. ₽ у «цены платежа»: тарифы не дешевели,
чуть сдвинулись их доли.

**Ответ:** одно число, рубли с двумя знаками — `round(x, 2)`.


In [ ]:
price = pd.Series({"basic": 199, "extended": 399})
share_before = pd.Series({"basic": 0.621, "extended": 0.379})
share_after = pd.Series({"basic": 0.557, "extended": 0.443})

answer = None   # ← ваш расчёт

check("★1", answer)


### ★ 2. Совместный член

Слайды 10–11. Когда меняются оба фактора сразу, часть изменения нельзя честно
отдать ни одному из них: `(n1 - n0) * (p1 - p0)`. В цепных подстановках она
достаётся фактору, который меняют вторым.

Посчитайте совместный член перехода октябрь → ноябрь в модели «активные подписки
× выручка на подписку». Значения берите из таблицы `m` как есть: `chain` округляет
до рубля и здесь не годится.

**Ответ:** одно число, рубли с двумя знаками — `round(x, 2)`.


In [ ]:
answer = None   # ← ваш расчёт

check("★2", answer)


### ★ 3. Среднее по шести порядкам

Слайды 11 и 13. У трёх факторов шесть порядков подстановки, и в каждом у числа
платежей на подписку своё влияние.

Посчитайте влияние фактора `per_sub` на переходе октябрь → ноябрь во всех шести
порядках и возьмите среднее. Функция `contrib` уже есть, порядки перечислит
`itertools.permutations`.

**Ответ:** одно число, рубли с двумя знаками — `round(x, 2)`.


In [ ]:
import itertools

x0 = m.loc["2025-10", ["active", "per_sub", "price"]].to_dict()
x1 = m.loc["2025-11", ["active", "per_sub", "price"]].to_dict()

answer = None   # ← ваш расчёт

check("★3", answer)


### ★ 4. `margins=True` тоже теряет клиентов

Слайды 3–4. У `pivot_table` есть аргумент `margins=True`: он дописывает строку
и колонку итогов `All`. Удобно — и опасно: итог считается только по тем строкам,
которые попали в таблицу.

Возьмите нетронутую копию `monthly_raw`: в ней пропуски канала на месте. Постройте
по ней сводную «канал × месяц» по выручке с `margins=True` двумя способами — как
есть и заполнив пропуск канала.

**Ответ:** список `[итог All за ноябрь как есть, итог All за ноябрь после
заполнения]`, оба числа с двумя знаками — `round(x, 2)`.


In [ ]:
answer = None   # ← ваш расчёт

check("★4", answer)


### ★ 5. Доли влияния в процентах

Слайд 16. Переведите два влияния перехода октябрь → ноябрь — активных подписок
и выручки на подписку — в проценты от изменения выручки. Порядок подстановок —
как в `chain`.

Посмотрите на результат и решите, понесли бы вы его директору.

**Ответ:** список из двух целых чисел `[доля активных подписок, доля выручки
на подписку]`. Каждое — `round(x)` от процентов, без второго аргумента.


In [ ]:
answer = None   # ← ваш расчёт

check("★5", answer)


### ★ 6. Где именно: канал × тариф

Слайды 5–6. Аддитивную модель можно резать по двум признакам сразу: клеток станет
четырнадцать, а сумма останется той же.

Посчитайте изменение выручки октябрь → ноябрь в каждой клетке «канал × тариф»
и верните три самые большие по модулю.

**Ответ:** список из трёх чисел по убыванию модуля, каждое с двумя знаками —
`round(x, 2)`. Знак сохраняйте.


In [ ]:
answer = None   # ← ваш расчёт

check("★6", answer)


### ★ 7. Самый спокойный переход года

Слайд 24. Загрузка в ячейке готова: та же таблица, но с февраля по декабрь
2025 года — одиннадцать месяцев, десять переходов. Январь не берём: история
начинается 6 января, месяц неполный.

Разложите каждый переход на активные подписки и выручку на подписку, как
в разделе 7. У какого перехода влияние выручки на подписку **ближе всего к нулю**?

Прежде чем считать, попробуйте угадать ответ по календарю. А потом посмотрите
на всю колонку: знак меняется вместе с длиной месяца.

**Ответ:** строка вида `"2025-06"` — месяц, **в который** пришли.


In [ ]:
year = load_monthly("true", "2025-02-01", "2025-12-01")      # идёт секунд десять

answer = None   # ← ваш расчёт

check("★7", answer)
